In [1]:
!pip install -q datasets transformers accelerate evaluate scikit-learn gensim gradio
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 58.1 MB/s eta 0:00:00
GPU available: True
Tesla T4


In [3]:
from huggingface_hub import list_datasets
for d in list_datasets(search="kinnews"):
    print(d.id)

andreniyongabo/kinnews_kirnews
Art3mis07/kinnews
KoseiUemura/KinNewsClassification
mteb/KinNewsClassification


In [5]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("mteb/KinNewsClassification", "kinnews_cleaned")
print(ds)

split = list(ds.keys())[0]
print(ds[split].features)
print(ds[split][0])

for s in ds.keys():
    df = ds[s].to_pandas()
    print(f"\n{s}: {len(df)} rows")
    print(df.columns.tolist())
    label_col = [c for c in df.columns if "label" in c.lower()][0]
    print(df[label_col].value_counts())

kinnews_cleaned/train-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 11.0MB            

kinnews_cleaned/train-00000-of-00001.par(…): downloading bytes:           |  0.00B            

kinnews_cleaned/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B / 2.44MB            

kinnews_cleaned/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9075 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2008 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9075
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2008
    })
})
{'text': Value('string'), 'label': Value('int64')}
{'text': 'bugesera nyuma y’inzara yaharanzwe karahahira utundi turere mu myaka yashize cyane cyane umwaka w’ akarere bugesera kagaragayemo ikibazo cy’amapfa akabije hari abantu bicwaga n’inzara abandi bagasuhukamuri gihe uhatembereye ushobora kugira amateka cyangwa se amakabyankuru y’abantu abenshi baturage hari bahatuye nibo bivugira kibazo cy’inzara kihaherutseniyonzima joseph w’imyaka utuye murenge nyamata akagari maranyundo amaze imyaka aka karere avuga inzara yatangiye kugaragara bugesera mpera y’ ariko ikaza gukaza umurego mwaka w’ avuga gihe ibintu byose byari byarabuze biryo mvaruganda kugeza bijumba bihingiragaagira umva ibijumba ibijumba gihe inaha byaribwaga n’umuntu ufite imodoka ndakubwiza ukuri mukene wapfaga kubyigondera niyonzima y

In [6]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

# ---- 1. Splits (dataset has no validation set, so carve 10% from train) ----
full_train = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()
train_df, val_df = train_test_split(
    full_train, test_size=0.1, stratify=full_train["label"], random_state=42
)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print("Sizes -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

# ---- 2. Full class distribution ----
dist = pd.concat(
    [d["label"].value_counts().rename(n) for n, d in
     [("train", train_df), ("val", val_df), ("test", test_df)]], axis=1
).fillna(0).astype(int).sort_index()
print("\nClass distribution:\n", dist)
print("\nNumber of classes:", dist.shape[0])

# ---- 3. Data quality checks ----
lens = train_df["text"].str.split().str.len()
print("\nWords per article:\n", lens.describe().round(1))
print("\nExact duplicate texts inside train:", train_df["text"].duplicated().sum())
print("Train/test text overlap (leakage):", len(set(train_df["text"]) & set(test_df["text"])))

# ---- 4. Baseline: TF-IDF + Logistic Regression ----
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000)
X_train = vec.fit_transform(train_df["text"])
X_val = vec.transform(val_df["text"])
X_test = vec.transform(test_df["text"])

clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)
clf.fit(X_train, train_df["label"])

for name, X, y in [("val", X_val, val_df["label"]), ("test", X_test, test_df["label"])]:
    p = clf.predict(X)
    print(f"\nBASELINE {name}: accuracy={accuracy_score(y, p):.4f} | macro-F1={f1_score(y, p, average='macro'):.4f}")

print("\nTest classification report:\n",
      classification_report(test_df["label"], clf.predict(X_test), digits=3, zero_division=0))

# ---- 5. Save splits for later steps ----
import os
os.makedirs("data", exist_ok=True)
train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

Sizes -> train: 8167 | val: 908 | test: 2008

Class distribution:
        train  val  test
label                  
0       1557  173   388
1       1312  146   313
2        954  106   228
3        993  111   256
4       1321  147   355
5        157   18    54
6        191   21    41
7         92   10     8
8        156   17    42
9         98   11    20
10       513   57   131
11       110   12    16
12       137   15    34
13       576   64   122

Number of classes: 14

Words per article:
 count    8167.0
mean      267.6
std       231.3
min         4.0
25%       155.5
50%       220.0
75%       317.0
max      8597.0
Name: text, dtype: float64

Exact duplicate texts inside train: 0
Train/test text overlap (leakage): 0

BASELINE val: accuracy=0.7786 | macro-F1=0.7246

BASELINE test: accuracy=0.7809 | macro-F1=0.7060

Test classification report:
               precision    recall  f1-score   support

           0      0.759     0.698     0.728       388
           1      0.946     0.946   

In [7]:
import pandas as pd
df = pd.read_csv("data/train.csv")
for lab in sorted(df["label"].unique()):
    print(f"\n=== LABEL {lab} ===")
    for t in df[df["label"] == lab]["text"].head(2):
        print("-", t[:200])


=== LABEL 0 ===
- motari yamenye atwaye amaze kwiba ibihumbi ahita amujyana polisi polisi y’ u rwanda yafashe abasore babiri bamaze kwiba amafaranga y’umucuruzi murenge kibeho karere nyaruguru polisi y’ u rwanda yafash
- gakenke inkeragutabara n’abafasha barahuguwe mateka kubakira byiza mateka y’u rwanda kugira uruhare kubungabunga umutekano hatangwa amakuru gihe muntu wese wahungabanya umutekano bimwe byagarutsweho m

=== LABEL 1 ===
- jose mulihno murugamba kugeza pedro ikipe chelsea umutoza w’ikipe chelsea jose mulihno yamaze gutangazako amaherezo pedro umukinnyi hagati wahariya i barcelone agomba hariya yerekeza ikipe chelsea ni 
- abakinnyi bamagare batanu berekeje isiganwa nyafurika abakinnyi batanu berekeje asmara adrien niyonshuti uherutse kwegukana tour de kigali’ gasore hategeka nicodem habiyambere nathan byukusenge joseph

=== LABEL 2 ===
- minisitiri w’intebe arashima abikorera rusizi bagiye kuzuza inzu y’amagorofa ane nubwo nzu kijyambere y’ubucuruzi rusizi rusizi commerc

In [8]:
import json, pandas as pd
from sklearn.metrics import classification_report

id2label = {0:"politics",1:"sport",2:"economy",3:"health",4:"entertainment",5:"history",
            6:"technology",7:"tourism",8:"culture",9:"fashion",10:"religion",
            11:"environment",12:"education",13:"relationship"}
label2id = {v:k for k,v in id2label.items()}
json.dump(id2label, open("data/id2label.json","w"))

test_df = pd.read_csv("data/test.csv")
pred = clf.predict(vec.transform(test_df["text"]))
print(classification_report(test_df["label"], pred,
      target_names=[id2label[i] for i in range(14)], digits=3, zero_division=0))

               precision    recall  f1-score   support

     politics      0.759     0.698     0.728       388
        sport      0.946     0.946     0.946       313
      economy      0.720     0.711     0.715       228
       health      0.708     0.691     0.700       256
entertainment      0.870     0.848     0.859       355
      history      0.481     0.481     0.481        54
   technology      0.756     0.829     0.791        41
      tourism      0.500     1.000     0.667         8
      culture      0.676     0.595     0.633        42
      fashion      0.789     0.750     0.769        20
     religion      0.873     0.893     0.883       131
  environment      0.368     0.438     0.400        16
    education      0.444     0.588     0.506        34
 relationship      0.736     0.893     0.807       122

     accuracy                          0.781      2008
    macro avg      0.688     0.740     0.706      2008
 weighted avg      0.784     0.781     0.781      2008



In [11]:
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

MODEL = "Davlan/afro-xlmr-base"
MAX_LEN = 256

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "label"]])
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text"])
    return ds.rename_column("label", "labels")

tr, va, te = to_ds(train_df), to_ds(val_df), to_ds(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=14, id2label=id2label, label2id=label2id)

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="afroxlmr_kinnews",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=150,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

print("\nVAL :", trainer.evaluate(va))
out = trainer.predict(te)
print("TEST:", out.metrics)

test_df["pred"] = out.predictions.argmax(-1)
test_df.to_csv("data/test_preds_afroxlmr.csv", index=False)
trainer.save_model("afroxlmr_final")
tok.save_pretrained("afroxlmr_final")

Map:   0%|          | 0/8167 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/2008 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.634437,0.772949,3,0.770925,0.687099



VAL : {'eval_loss': 0.7729493379592896, 'eval_accuracy': 0.7709251101321586, 'eval_macro_f1': 0.6870992741868338}


TEST: {'test_loss': 0.7683781981468201, 'test_accuracy': 0.7788844621513944, 'test_macro_f1': 0.6566530220329557, 'test_runtime': 7.1923, 'test_samples_per_second': 279.187, 'test_steps_per_second': 8.759}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('afroxlmr_final/tokenizer_config.json', 'afroxlmr_final/tokenizer.json')

In [12]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json
BK = "/content/drive/MyDrive/kinnews_capstone"
os.makedirs(BK, exist_ok=True)
shutil.copytree("data", f"{BK}/data", dirs_exist_ok=True)
shutil.copytree("afroxlmr_final", f"{BK}/afroxlmr_final_exp1", dirs_exist_ok=True)

results = [
 {"name": "baseline_tfidf_logreg", "val_macro_f1": 0.7246, "test_acc": 0.7809, "test_macro_f1": 0.7060},
 {"name": "exp1_afroxlmr_3ep_lr2e-5", "val_macro_f1": 0.6871, "test_acc": 0.7789, "test_macro_f1": 0.6567},
]
json.dump(results, open(f"{BK}/results.json", "w"), indent=2)
print("Backed up to", BK)

Mounted at /content/drive
Backed up to /content/drive/MyDrive/kinnews_capstone


In [13]:
import gc, math
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight("balanced", classes=np.arange(14), y=train_df["label"].values)
cw_tensor = torch.tensor(cw, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

def run_experiment(name, epochs=3, lr=2e-5, weighted=False, max_len=256, bs=16):
    def make(df):
        ds = Dataset.from_pandas(df[["text", "label"]])
        ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                    batched=True, remove_columns=["text"])
        return ds.rename_column("label", "labels")
    tr_, va_, te_ = make(train_df), make(val_df), make(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=14, id2label=id2label, label2id=label2id)
    steps = math.ceil(len(train_df) / bs) * epochs
    args = TrainingArguments(
        output_dir=f"runs/{name}", learning_rate=lr,
        per_device_train_batch_size=bs, per_device_eval_batch_size=32,
        num_train_epochs=epochs, weight_decay=0.01, warmup_steps=int(0.1 * steps),
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", fp16=torch.cuda.is_available(),
        save_total_limit=1, report_to="none", seed=42)
    cls = WeightedTrainer if weighted else Trainer
    kw = {"class_weights": cw_tensor} if weighted else {}
    trainer = cls(model=model, args=args, train_dataset=tr_, eval_dataset=va_,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics, **kw)
    trainer.train()

    v = trainer.evaluate(va_)
    out = trainer.predict(te_)
    res = {"name": name, "epochs": epochs, "lr": lr, "weighted": weighted, "max_len": max_len,
           "val_macro_f1": round(v["eval_macro_f1"], 4),
           "test_acc": round(out.metrics["test_accuracy"], 4),
           "test_macro_f1": round(out.metrics["test_macro_f1"], 4)}
    results.append(res)
    json.dump(results, open(f"{BK}/results.json", "w"), indent=2)

    tdf = test_df.copy(); tdf["pred"] = out.predictions.argmax(-1)
    tdf.to_csv(f"{BK}/data/test_preds_{name}.csv", index=False)
    trainer.save_model(f"models/{name}"); tok.save_pretrained(f"models/{name}")

    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    print("\nRESULT:", res)
    return res

run_experiment("exp2_6ep_lr2e-5", epochs=6)

Map:   0%|          | 0/8167 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/2008 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.351042,0.770546,6,0.790749,0.729796


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RESULT: {'name': 'exp2_6ep_lr2e-5', 'epochs': 6, 'lr': 2e-05, 'weighted': False, 'max_len': 256, 'val_macro_f1': 0.7298, 'test_acc': 0.7754, 'test_macro_f1': 0.675}


{'name': 'exp2_6ep_lr2e-5',
 'epochs': 6,
 'lr': 2e-05,
 'weighted': False,
 'max_len': 256,
 'val_macro_f1': 0.7298,
 'test_acc': 0.7754,
 'test_macro_f1': 0.675}

In [14]:
run_experiment("exp3_6ep_weighted", epochs=6, weighted=True)

Map:   0%|          | 0/8167 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/2008 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.381934,1.041140,6,0.784141,0.722265


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RESULT: {'name': 'exp3_6ep_weighted', 'epochs': 6, 'lr': 2e-05, 'weighted': True, 'max_len': 256, 'val_macro_f1': 0.7223, 'test_acc': 0.7804, 'test_macro_f1': 0.6894}


{'name': 'exp3_6ep_weighted',
 'epochs': 6,
 'lr': 2e-05,
 'weighted': True,
 'max_len': 256,
 'val_macro_f1': 0.7223,
 'test_acc': 0.7804,
 'test_macro_f1': 0.6894}

In [15]:
shutil.copytree("models/exp2_6ep_lr2e-5", f"{BK}/model_exp2_best", dirs_exist_ok=True)
shutil.copytree("models/exp3_6ep_weighted", f"{BK}/model_exp3", dirs_exist_ok=True)
print(os.listdir(BK))

['data', 'afroxlmr_final_exp1', 'results.json', 'model_exp2_best', 'model_exp3']


In [16]:
run_experiment("exp4_6ep_len512", epochs=6, max_len=512, bs=8)

Map:   0%|          | 0/8167 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/2008 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.302534,0.887737,6,0.794053,0.738986


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RESULT: {'name': 'exp4_6ep_len512', 'epochs': 6, 'lr': 2e-05, 'weighted': False, 'max_len': 512, 'val_macro_f1': 0.739, 'test_acc': 0.7908, 'test_macro_f1': 0.6907}


{'name': 'exp4_6ep_len512',
 'epochs': 6,
 'lr': 2e-05,
 'weighted': False,
 'max_len': 512,
 'val_macro_f1': 0.739,
 'test_acc': 0.7908,
 'test_macro_f1': 0.6907}

In [17]:
shutil.copytree("models/exp4_6ep_len512", f"{BK}/model_exp4_best", dirs_exist_ok=True)

from sklearn.metrics import classification_report, confusion_matrix, f1_score
df = pd.read_csv(f"{BK}/data/test_preds_exp4_6ep_len512.csv")
names = [id2label[i] for i in range(14)]
print(classification_report(df["label"], df["pred"], target_names=names, digits=3, zero_division=0))

# per-class F1: baseline vs final model
base_pred = clf.predict(vec.transform(df["text"]))
cmp = pd.DataFrame({
    "n_test": df["label"].value_counts().reindex(range(14)).values,
    "baseline_f1": f1_score(df["label"], base_pred, labels=range(14), average=None),
    "exp4_f1": f1_score(df["label"], df["pred"], labels=range(14), average=None),
}, index=names).round(3)
cmp["diff"] = (cmp["exp4_f1"] - cmp["baseline_f1"]).round(3)
print("\nPer-class F1 comparison:\n", cmp)

# most common confusions
cm = confusion_matrix(df["label"], df["pred"], labels=range(14))
pairs = [(id2label[i], id2label[j], int(cm[i, j])) for i in range(14) for j in range(14) if i != j and cm[i, j] > 0]
print("\nTop confusions:\n", pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:12],
                                         columns=["true", "predicted", "count"]))

# does article length matter?
df["words"] = df["text"].str.split().str.len()
df["correct"] = df["label"] == df["pred"]
print("\nMean words, correct vs wrong:\n", df.groupby("correct")["words"].mean().round(1))

# misclassified examples
wrong = df[~df["correct"]].copy()
wrong["true"] = wrong["label"].map(id2label)
wrong["predicted"] = wrong["pred"].map(id2label)
wrong[["true", "predicted", "words", "text"]].to_csv(f"{BK}/data/misclassified_exp4.csv", index=False)
print(f"\nMisclassified: {len(wrong)} of {len(df)}\n")
for _, r in wrong.sample(20, random_state=1).iterrows():
    print(f"[true={r['true']} | pred={r['predicted']} | {r['words']} words]\n  {r['text'][:250]}\n")

               precision    recall  f1-score   support

     politics      0.761     0.745     0.753       388
        sport      0.935     0.958     0.946       313
      economy      0.724     0.702     0.713       228
       health      0.721     0.707     0.714       256
entertainment      0.885     0.893     0.889       355
      history      0.545     0.444     0.490        54
   technology      0.759     0.537     0.629        41
      tourism      0.444     1.000     0.615         8
      culture      0.541     0.476     0.506        42
      fashion      0.789     0.750     0.769        20
     religion      0.833     0.916     0.873       131
  environment      0.389     0.438     0.412        16
    education      0.514     0.529     0.522        34
 relationship      0.805     0.877     0.839       122

     accuracy                          0.791      2008
    macro avg      0.689     0.712     0.691      2008
 weighted avg      0.789     0.791     0.789      2008


Per-cl

In [23]:
!pip install -q gradio transformers sentencepiece
import os, pandas as pd, torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Bianca255/kinnews-afroxlmr"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()

def classify(text):
    text = (text or "").lower().strip()  # training data was lowercased
    if not text:
        return {}
    enc = tok(text, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0].cpu()
    return {model.config.id2label[i]: float(p) for i, p in enumerate(probs)}

# example inputs: first 100 words of a sport, religion and relationship article
examples = []
for p in ["data/test.csv", "/content/drive/MyDrive/kinnews_capstone/data/test.csv"]:
    if os.path.exists(p):
        t = pd.read_csv(p)
        examples = [[" ".join(t[t["label"] == l]["text"].iloc[0].split()[:100])] for l in (1, 10, 13)]
        break

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=8, label="Kinyarwanda news text"),
    outputs=gr.Label(num_top_classes=5, label="Predicted topic"),
    title="Kinyarwanda News Topic Classifier",
    description="AfroXLMR fine-tuned on KINNEWS (14 topics). It always picks one of the 14 topics, even for text outside the news domain.",
    examples=examples,
    cache_examples=False,
)
demo.launch(share=True)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee269f46338b6071d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
